# Notebook 9 – Feature Selection

This notebook covers how to pick the most useful features for a Machine Learning model, and how to remove features that add noise instead of value.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


**Code Explanation:** The dataset is loaded, rows without a CustomerID are dropped since they are not usable for customer-level analysis, and a new TotalPrice column is created by multiplying Quantity and UnitPrice.

## 1. What is Feature Selection?

**Definition:** The process of choosing the most useful columns from a dataset and dropping the ones that do not help the model.

**Example:** Out of 20 columns, only 6 might actually help predict whether a customer will churn.

**Why it is used?** Fewer, better features make models faster, easier to understand, and often more accurate.

In [2]:
features = df[['Quantity', 'UnitPrice', 'TotalPrice']]
features.describe()

,Quantity,UnitPrice,TotalPrice
count,406829.000000,406829.000000,406829.000000
mean,12.061303,3.460471,20.401854
std,248.693370,69.315162,427.591718
min,-80995.000000,0.000000,-168469.600000
25%,2.000000,1.250000,4.200000
50%,5.000000,1.950000,11.100000
75%,12.000000,3.750000,19.500000
max,80995.000000,38970.000000,168469.600000


**Code Explanation:** A small feature set is created from the numeric columns, and describe gives a quick summary of their ranges and spread.

## 2. Why Feature Selection?

**Definition:** Feature selection is needed because not every column in a dataset actually helps a model make better predictions.

**Example:** A random ID column will not help predict sales, even though it is a number.

**Why it is used?** Removes noise, reduces overfitting, shortens training time, and makes the model easier to interpret.

In [3]:
correlation = df[['Quantity', 'UnitPrice', 'TotalPrice']].corr()
correlation

,Quantity,UnitPrice,TotalPrice
Quantity,1.000000,-0.001238,0.916096
UnitPrice,-0.001238,1.000000,-0.129296
TotalPrice,0.916096,-0.129296,1.000000


**Code Explanation:** corr calculates how strongly each numeric feature relates to the others, which is a first step toward deciding what to keep.

## 3. Relevant Features

**Definition:** Features that have a real, meaningful relationship with the target variable.

**Example:** UnitPrice is relevant when predicting TotalPrice, since TotalPrice is directly built from it.

**Why it is used?** Keeping relevant features ensures the model has the actual signal it needs to learn from.

In [4]:
df[['UnitPrice', 'TotalPrice']].corr()

,UnitPrice,TotalPrice
UnitPrice,1.000000,-0.129296
TotalPrice,-0.129296,1.000000


**Code Explanation:** The correlation between UnitPrice and TotalPrice is checked to confirm how strongly they are related.

## 4. Irrelevant Features

**Definition:** Features that have little or no relationship with the target and do not help predictions.

**Example:** Country name is unlikely to meaningfully predict TotalPrice on its own.

**Why it is used?** Identifying and removing irrelevant features keeps the model focused on signals that actually matter.

In [5]:
df.groupby('Country')['TotalPrice'].mean().sort_values(ascending=False).head()

Country
Netherlands    120.059696
Australia      108.877895
Japan           98.716816
Sweden          79.211926
Denmark         48.247147
Name: TotalPrice, dtype: float64

**Code Explanation:** Average TotalPrice is grouped by Country to check whether Country shows any strong or meaningful pattern.

## 5. Redundant Features

**Definition:** Features that repeat information already captured by another feature.

**Example:** TotalPrice is redundant if Quantity and UnitPrice are already in the dataset, since it is just their product.

**Why it is used?** Removing redundant features avoids double-counting the same information and keeps the feature set clean.

In [6]:
df[['Quantity', 'UnitPrice', 'TotalPrice']].corr()['TotalPrice']

Quantity      0.916096
UnitPrice    -0.129296
TotalPrice    1.000000
Name: TotalPrice, dtype: float64

**Code Explanation:** Correlation of TotalPrice with Quantity and UnitPrice is checked to confirm it is largely built from those two columns.

## 6. Filter Methods

**Definition:** Feature selection techniques that score each feature using statistics, without involving any model.

**Example:** Ranking features by correlation or variance before training anything.

**Why it is used?** Fast and simple, good for a first pass on large datasets.

**Advantages:** Very fast, model-independent, easy to understand.
**Limitations:** Ignores interactions between features and may miss features that only matter when combined.

In [7]:
variances = df[['Quantity', 'UnitPrice', 'TotalPrice']].var()
variances

Quantity       61848.392291
UnitPrice       4804.591645
TotalPrice    182834.676918
dtype: float64

**Code Explanation:** var calculates how much each feature varies, which is a common filter-based scoring method.

## 7. Wrapper Methods

**Definition:** Feature selection techniques that use a model to test different subsets of features and pick the best-performing combination.

**Example:** Trying different feature combinations and checking which one gives the best accuracy.

**Why it is used?** Considers how features work together, not just individually.

**Advantages:** Usually more accurate since it accounts for feature interactions.
**Limitations:** Slow and computationally expensive, especially with many features.

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
X = df[['Quantity', 'UnitPrice']]
y = df['TotalPrice']
model = LinearRegression()
rfe = RFE(model, n_features_to_select=1)
rfe.fit(X, y)
rfe.support_

array([ True, False])

**Code Explanation:** RFE wraps around a LinearRegression model and repeatedly removes the weakest feature until only 1 feature remains, showing which one matters most.

## 8. Embedded Methods

**Definition:** Feature selection that happens automatically as part of training a model.

**Example:** A Lasso regression model shrinking unimportant feature weights to zero while training.

**Why it is used?** Combines the speed of filter methods with the accuracy benefits of wrapper methods.

**Advantages:** Efficient and built into the model training process.
**Limitations:** Tied to a specific model, so results may not transfer well to a different algorithm.

In [9]:
from sklearn.linear_model import Lasso
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)
lasso.coef_

array([ 1.57481821, -0.79059045])

**Code Explanation:** Lasso is trained on the features, and its coefficients show which features it considered useful, since less useful ones shrink toward zero.

## 9. Variance Threshold

**Definition:** A filter method that removes features with very low variance, meaning features that barely change across rows.

**Example:** A column where 99 percent of the values are the same number adds almost no useful information.

**Why it is used?** Quickly eliminates near-constant features before deeper analysis.

**Advantages:** Very fast and simple to apply.
**Limitations:** Only looks at spread, not at the relationship with the target, so it can miss important low-variance features.

In [10]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.1)
selected = selector.fit_transform(df[['Quantity', 'UnitPrice', 'TotalPrice']])
selector.get_support()

array([ True,  True,  True])

**Code Explanation:** VarianceThreshold removes any feature whose variance is below 0.1, and get_support shows which features passed the check.

## 10. Correlation-Based Selection

**Definition:** Selecting features by measuring how strongly each one correlates with the target, and dropping highly correlated features with each other.

**Example:** Dropping one of two features if they have a correlation above 0.9 with each other.

**Why it is used?** Reduces redundancy and multicollinearity in the feature set.

**Advantages:** Simple to compute and interpret.
**Limitations:** Only captures linear relationships, so it can miss non-linear patterns.

In [12]:
corr_matrix = df[['Quantity', 'UnitPrice', 'TotalPrice']].corr().abs()
corr_matrix

,Quantity,UnitPrice,TotalPrice
Quantity,1.000000,0.001238,0.916096
UnitPrice,0.001238,1.000000,0.129296
TotalPrice,0.916096,0.129296,1.000000


**Code Explanation:** The absolute correlation between all numeric features is calculated to spot pairs that are too similar to each other.

## 11. Mutual Information

**Definition:** A measure of how much knowing one variable reduces uncertainty about another, capturing both linear and non-linear relationships.

**Example:** Mutual information can detect that Quantity affects TotalPrice even if the relationship is not a straight line.

**Why it is used?** Captures more complex relationships than plain correlation.

**Advantages:** Works for non-linear relationships and different data types.
**Limitations:** Harder to interpret and more computationally expensive than correlation.

In [13]:
from sklearn.feature_selection import mutual_info_regression
mi_scores = mutual_info_regression(df[['Quantity', 'UnitPrice']], df['TotalPrice'])
mi_scores

array([2.2910949 , 3.12171428])

**Code Explanation:** mutual_info_regression scores how much each feature contributes to predicting TotalPrice, including non-linear effects.

## 12. Chi-Square

**Definition:** A statistical test used to check whether a categorical feature is independent of a categorical target.

**Example:** Testing whether Country is related to whether a customer is a repeat buyer.

**Why it is used?** Good for selecting categorical features in classification problems.

**Advantages:** Well suited for categorical data and easy to interpret.
**Limitations:** Only works with non-negative categorical or binned data, not raw continuous values.

In [14]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder
df['Country_Code'] = LabelEncoder().fit_transform(df['Country'])
df['High_Value'] = (df['TotalPrice'] > df['TotalPrice'].median()).astype(int)
chi_scores, p_values = chi2(df[['Country_Code']], df['High_Value'])
chi_scores

array([18441.35656239])

**Code Explanation:** Country is label-encoded into numbers, a High_Value target is created based on the median TotalPrice, and chi2 tests whether Country relates to High_Value.

## 13. Recursive Feature Elimination (RFE)

**Definition:** A wrapper method that repeatedly trains a model, removes the weakest feature, and repeats until the desired number of features remains.

**Example:** Starting with 10 features and removing one at a time until only 3 are left.

**Why it is used?** Finds a strong feature subset by testing actual model performance.

**Advantages:** Considers feature interactions and usually gives a strong final feature set.
**Limitations:** Computationally expensive since it retrains the model many times.

In [15]:
from sklearn.linear_model import LogisticRegression
X = df[['Quantity', 'UnitPrice', 'Country_Code']]
y = df['High_Value']
rfe_model = RFE(LogisticRegression(max_iter=1000), n_features_to_select=2)
rfe_model.fit(X, y)
rfe_model.support_

array([ True,  True, False])

**Code Explanation:** RFE is applied with a LogisticRegression model to select the 2 strongest features out of the 3 available for predicting High_Value.

## 14. L1/Lasso-Based Selection

**Definition:** Using L1 regularization to shrink less important feature coefficients to exactly zero, effectively removing them.

**Example:** A Lasso model setting the coefficient of a weak feature to 0 while keeping strong ones non-zero.

**Why it is used?** Naturally combines model training with feature selection.

**Advantages:** Efficient, automatic, and works well with many features.
**Limitations:** Assumes roughly linear relationships and can behave unpredictably when features are highly correlated with each other.

In [16]:
from sklearn.linear_model import Lasso
lasso_model = Lasso(alpha=0.5)
lasso_model.fit(df[['Quantity', 'UnitPrice', 'Country_Code']], df['TotalPrice'])
lasso_model.coef_

array([ 1.57477836, -0.79058182, -0.13086861])

**Code Explanation:** Lasso is trained with a higher alpha value to push weaker feature coefficients toward zero, revealing which features it kept.

## 15. Tree-Based Feature Importance

**Definition:** Using a tree-based model like Random Forest to rank features by how much they help split the data correctly.

**Example:** A Random Forest showing that UnitPrice contributes more to predictions than Country.

**Why it is used?** Captures non-linear relationships and feature interactions naturally.

**Advantages:** Handles non-linear data well and gives an intuitive importance ranking.
**Limitations:** Importance scores can be biased toward features with many unique values, and results can vary between runs.

In [17]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(df[['Quantity', 'UnitPrice', 'Country_Code']], df['TotalPrice'])
rf_model.feature_importances_

array([8.68231210e-01, 1.31641571e-01, 1.27219248e-04])

**Code Explanation:** A Random Forest is trained on the features, and feature_importances_ shows how much each feature contributed to the model's decisions.

## Advantages and Limitations Summary

**Filter Methods**
Advantage: fast and simple. Limitation: ignores feature interactions.

**Wrapper Methods**
Advantage: accounts for feature combinations. Limitation: slow and expensive.

**Embedded Methods**
Advantage: built into model training, efficient. Limitation: tied to the specific model used.

**Variance Threshold**
Advantage: quick way to drop near-constant columns. Limitation: ignores the target variable entirely.

**Correlation-Based Selection**
Advantage: simple and interpretable. Limitation: misses non-linear relationships.

**Mutual Information**
Advantage: captures non-linear relationships. Limitation: more complex to compute and interpret.

**Chi-Square**
Advantage: effective for categorical features. Limitation: not usable directly on continuous data.

**Recursive Feature Elimination**
Advantage: strong final feature subset. Limitation: computationally heavy for large feature sets.

**L1/Lasso-Based Selection**
Advantage: automatic and efficient. Limitation: less reliable with highly correlated features.

**Tree-Based Feature Importance**
Advantage: handles non-linear patterns well. Limitation: can be biased and inconsistent across runs.